 # 2.08c — Final Logistic Classifier Training on All Data

 In 2.05 I trained the Logistic classifiers on 2019+2021 and held 2026 out for
 evaluation. Here I refit all 6 on the full 2019 + 2021 + 2026 dataset. The AUC and
 Brier scores from 2.05 are the honest numbers and do not change.

 I am using the same saga config from 2.05: penalty=None because L2 is essentially
 inert at 14M rows, and tol=1e-2 because that is the tolerance that actually triggered
 convergence in about 40 epochs. Running max_iter=200 with tol=1e-3 never stopped
 early and ground through all 200 epochs every time, so tol=1e-2 is the right call.

 A single full-data saga fit is fine on 14M rows. The OOM problem in 2.05 only came up
 with CV folds where the training set hit 10M+ rows and the dense matrix grew to several
 GB. One fit without cross-validation does not have that issue because saga streams
 stochastic updates rather than materializing the full matrix all at once.

 This is the last of the three final-training notebooks. The verification cell at the
 bottom loads all 18 artifacts and runs a one-row predict on each one as a sanity check.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path().resolve().parent))
from model_training.feature_prep import (
    TARGET_CLASSIFICATION,
    TARGET_REGRESSION,
    TRAINING_YEARS,
    build_preprocessor,
    get_X_y,
    load_training_data,
)

 ## Setup

 Same horizons as 2.05. FULL_YEARS includes 2026 with nothing held out.

In [ ]:
HORIZONS = [60, 180, 360, 720, 1440, 2880]
HORIZON_LABELS = {60: "1hr", 180: "3hr", 360: "6hr",
                  720: "12hr", 1440: "24hr", 2880: "multi-day"}

FULL_YEARS  = TRAINING_YEARS   # [2019, 2021, 2026], no holdout withheld
USE_FLOAT32 = True

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Horizons: {[HORIZON_LABELS[h] for h in HORIZONS]}")
print(f"Training on: {FULL_YEARS}")
print(f"Artifacts -> {MODELS_DIR.resolve()}")

Horizons: ['1hr', '3hr', '6hr', '12hr', '24hr', 'multi-day']
Training on: [2019, 2021, 2026]
Artifacts -> C:\Users\clark\Desktop\citibike\models


 ## Hyperparameters

 The saga config from 2.05, hardcoded here. penalty=None because at 12M+ rows L2 adds
 bias without meaningfully reducing variance. class_weight=None so the predicted
 probabilities stay calibrated rather than being pushed toward the minority class.

In [ ]:
BASE_LOGIT_PARAMS = dict(
    penalty      = None,
    solver       = "saga",
    max_iter     = 200,
    tol          = 1e-2,
    class_weight = None,
    random_state = 42,
    verbose      = 0,
)

 ## Train all 6 horizons

 One horizon at a time. Each fit loads about 14M rows, runs saga to convergence, saves
 the bare pipeline, then frees memory. The serving layer calls predict_proba(X)[:, 1]
 to get the probability that a bike is available.

In [ ]:
for h in HORIZONS:
    label = HORIZON_LABELS[h]
    t0 = time.time()

    df = load_training_data(h, years=FULL_YEARS)
    X, y = get_X_y(df, TARGET_CLASSIFICATION)
    y = y.astype("int8")
    if USE_FLOAT32:
        num_cols = X.select_dtypes(include=["float64", "float32"]).columns
        X[num_cols] = X[num_cols].astype("float32")
    base_rate = float(np.mean(y))
    span = f"{df['timestamp'].min().date()} -> {df['timestamp'].max().date()}"
    del df

    print(f"  {label:<10} {len(X):>11,} rows  base rate {base_rate:.4f}  ({span})  fitting ...",
          end=" ", flush=True)

    pipe = Pipeline([
        ("pre",   build_preprocessor("linear")),
        ("model", LogisticRegression(**BASE_LOGIT_PARAMS)),
    ])
    pipe.fit(X, y)

    out = MODELS_DIR / f"logistic_classification_{h}min.joblib"
    joblib.dump(pipe, out)
    print(f"saved -> {out.name}   ({time.time() - t0:.0f}s)")
    del X, y, pipe

  1hr         14,674,158 rows  base rate 0.9358  (2019-01-01 -> 2026-06-18)  fitting ... saved -> logistic_classification_60min.joblib   (1323s)
  3hr         14,355,238 rows  base rate 0.9357  (2019-01-01 -> 2026-06-17)  fitting ... saved -> logistic_classification_180min.joblib   (956s)
  6hr         14,007,554 rows  base rate 0.9361  (2019-01-01 -> 2026-06-17)  fitting ... saved -> logistic_classification_360min.joblib   (940s)
  12hr        13,722,578 rows  base rate 0.9352  (2019-01-01 -> 2026-06-17)  fitting ... saved -> logistic_classification_720min.joblib   (970s)
  24hr        14,680,390 rows  base rate 0.9361  (2019-01-01 -> 2026-06-17)  fitting ... saved -> logistic_classification_1440min.joblib   (1022s)
  multi-day   14,476,193 rows  base rate 0.9359  (2019-01-01 -> 2026-06-16)  fitting ... saved -> logistic_classification_2880min.joblib   (997s)


 ## Verify all 18 artifacts

 Now that all three notebooks have run, I want to make sure every artifact loads and
 produces a sensible prediction before I call this done. I pull one row from the 1hr
 horizon as a dummy input and pass it through all 18 pipelines. Anything missing or
 mis-shaped will show up here rather than in production.

In [ ]:
print("\nVerifying all 18 artifacts ...")
df_demo = load_training_data(60, years=FULL_YEARS)
X_demo, _ = get_X_y(df_demo, TARGET_REGRESSION)
x_one = X_demo.iloc[[0]]
del df_demo, X_demo

ok = 0
for h in HORIZONS:
    label = HORIZON_LABELS[h]

    lg = joblib.load(MODELS_DIR / f"lgbm_regression_{h}min.joblib")
    assert isinstance(lg, Pipeline), f"lgbm {label} is not a Pipeline"
    lg_pred = float(lg.predict(x_one)[0])

    lin = joblib.load(MODELS_DIR / f"linear_regression_{h}min.joblib")
    assert set(lin.keys()) == {"pipeline", "pi_lower_offset", "pi_upper_offset", "pi_level"}, \
        f"linear {label} has wrong keys: {set(lin.keys())}"
    lin_pt = float(lin["pipeline"].predict(x_one)[0])
    lin_lo = max(0.0, lin_pt + lin["pi_lower_offset"])
    lin_hi = lin_pt + lin["pi_upper_offset"]

    clf = joblib.load(MODELS_DIR / f"logistic_classification_{h}min.joblib")
    assert isinstance(clf, Pipeline), f"logistic {label} is not a Pipeline"
    clf_p = float(clf.predict_proba(x_one)[0, 1])

    print(f"  {label:<10} lgbm {lg_pred:6.2f}   "
          f"linear {lin_pt:6.2f} [{lin_lo:5.1f}, {lin_hi:5.1f}]   "
          f"P(bike) {clf_p:.3f}")
    ok += 3

print(f"\nAll {ok} artifacts loaded and produced a prediction.")


Verifying all 18 artifacts ...
  1hr        lgbm  36.04   linear  33.30 [ 26.7,  40.1]   P(bike) 1.000
  3hr        lgbm  37.27   linear  28.44 [ 16.7,  40.8]   P(bike) 0.999
  6hr        lgbm  34.07   linear  24.80 [  9.5,  40.7]   P(bike) 0.989
  12hr       lgbm  35.65   linear  22.39 [  6.9,  39.0]   P(bike) 0.989
  24hr       lgbm  27.10   linear  26.58 [ 11.5,  42.7]   P(bike) 0.995
  multi-day  lgbm  21.06   linear  23.70 [  6.8,  42.1]   P(bike) 0.982

All 18 artifacts loaded and produced a prediction.


 ## Conclusion

 All 18 production artifacts are trained on the full 2019 + 2021 + 2026 dataset and
 written to models/. The hyperparameters are frozen from 2.02 and 2.05. The Linear
 prediction intervals are the holdout-calibrated offsets from 2.04b, reused unchanged.
 The performance numbers to quote are in 2.04 (LightGBM RMSE/MAE), 2.04b (Linear
 RMSE/MAE and PI coverage), and 2.05 (Logistic AUC and Brier). This closes out Phase 3.5.